# FUSION 2026 Multi-Target Figures

Refactored notebook with deterministic orchestration and reusable pipeline helpers.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from fusion2026_mt_pipeline import (
    build_config,
    build_platform,
    build_plugin_vs_ss_figure,
    build_simulator,
    build_target_truths,
    build_timesteps,
    build_tracker_figure,
    build_world_figure,
    generate_stonesoup_detections,
    run_detection,
    run_tracking_jpda,
    write_figures,
)

/home/fin/miniconda3/envs/nereus-env/lib/python3.12/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


## 1) Configuration


In [3]:
cfg = build_config(seed=12)
print(f"Total simulation duration: {cfg['total_duration_s']} seconds")
print(
    f"Number of timesteps: {cfg['sim']['num_steps']}, "
    f"Timestep interval: {cfg['sim']['time_interval'].total_seconds()} seconds"
)
print(f"Num targets: {len(cfg['targets'])}")

Total simulation duration: 900.0 seconds
Number of timesteps: 180, Timestep interval: 5.0 seconds
Num targets: 3


## 2) Scenario Build + Detection


In [4]:
platform = build_platform(cfg)
target_ground_truths, relative_bearing_ground_truths = build_target_truths(cfg, platform)
simulator = build_simulator(cfg, platform, target_ground_truths)

print("Running detection chain...")
all_detections, snr_map = run_detection(cfg, simulator)
timesteps = build_timesteps(cfg)

Running detection chain...


Generating Detections: 180it [01:38,  1.83it/s]


## 3) Tracking + Stone Soup Baseline


In [5]:
_, all_tracks = run_tracking_jpda(cfg, all_detections, relative_bearing_ground_truths)
stone_soup_detections = generate_stonesoup_detections(
    cfg, timesteps, target_ground_truths, platform
)

## 4) Build Figures


In [6]:
world_fig = build_world_figure(platform, target_ground_truths)
tracker_fig = build_tracker_figure(
    timesteps=timesteps,
    steering_azimuths_rad=cfg["beamforming"]["steering_azimuths_rad"],
    snr_map=snr_map,
    all_detections=all_detections,
    all_tracks=all_tracks,
    relative_bearing_ground_truths=relative_bearing_ground_truths,
)
plugin_vs_ss_fig = build_plugin_vs_ss_figure(timesteps, all_detections, stone_soup_detections)

world_fig.show()
tracker_fig.show()
plugin_vs_ss_fig.show()

## 5) Export


In [7]:
write_figures(
    {
        "mt_world_picture.pdf": world_fig,
        "mt_bf_tracker.pdf": tracker_fig,
        "mt_plugin_vs_ss.pdf": plugin_vs_ss_fig,
    },
    output_dir="figs",
)